# Аналитика по листу «Заказы белые ТЕСТ»

Ноутбук читает данные из Google Sheets таблицы `Расчет поставки Китай_по обороту`, лист `Заказы белые ТЕСТ`, и формирует аналитический датафрейм `df_orders_white_test` по контролируемому списку колонок.

In [1]:
from datetime import datetime
from pathlib import Path
import re
import sys

import pandas as pd
from IPython.display import display

# Делаем импорты проекта устойчивыми при запуске из корня проекта или из папки polygon.
project_root = Path.cwd().resolve()
if project_root.name == "polygon":
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

project_root

WindowsPath('C:/Users/123/Desktop/start_vector')

In [4]:
import logging
import re
from datetime import datetime

import pandas as pd

from src_oop.core.my_gspread import GoogleTabs
from src_oop.core.utils_general import clean_currency_value
from src_oop.jobs.calculation_of_purchases_china.config import (
    LOGISTICS_VED_REQUIRED_COLUMNS,
    ORDERS_WHITE_AVANS_COLS,
    ORDERS_WHITE_BALANCE_ONE_COLS,
    ORDERS_WHITE_BALANCE_TWO_COLS,
    ORDERS_WHITE_CANCELED_STATUSES,
    ORDERS_WHITE_DIGIT_COLS,
    ORDERS_WHITE_PAYMENT_CONFIGS,
    ORDERS_WHITE_REQUIRED_COLUMNS,
    ORDERS_WHITE_UPDATED_AT_COLUMN,
    delivery_calculation_china,
)

logger = logging.getLogger(__name__)


class OrdersWhiteBalanceAnalyticsService:
    """Формирует и выгружает аналитику платежей по листу заказов белых."""

    def __init__(self) -> None:
        self._table_name = delivery_calculation_china.get("title")
        self._source_sheet_name = delivery_calculation_china.get("white_orders_sheet")
        self._target_sheet_name = delivery_calculation_china.get("payments_analyze_sheet")

        self._source_conn: GoogleTabs | None = None
        self._target_conn: GoogleTabs | None = None

    @property
    def source_connect(self) -> GoogleTabs:
        if self._source_conn is None:
            self._source_conn = GoogleTabs(
                table_title=self._table_name,
                sheet_title=self._source_sheet_name,
            )
        return self._source_conn

    @property
    def target_connect(self) -> GoogleTabs:
        if self._target_conn is None:
            self._target_conn = GoogleTabs(
                table_title=self._table_name,
                sheet_title=self._target_sheet_name,
            )
        return self._target_conn

    @staticmethod
    def normalize_column_name(column_name: object) -> str:
        normalized = str(column_name).replace("\t", " ").replace("\n", " ").replace("\r", " ")
        return re.sub(r"\s+", " ", normalized).strip()

    @staticmethod
    def make_unique_column_names(column_names: list[str]) -> list[str]:
        seen_columns: dict[str, int] = {}
        unique_columns: list[str] = []

        for column_name in column_names:
            seen_columns[column_name] = seen_columns.get(column_name, 0) + 1
            if seen_columns[column_name] == 1:
                unique_columns.append(column_name)
                continue

            unique_columns.append(f"{column_name} {seen_columns[column_name]}")

        return unique_columns

    @staticmethod
    def validate_required_columns(df: pd.DataFrame, required_columns: list[str]) -> None:
        missing_columns = [column for column in required_columns if column not in df.columns]
        if missing_columns:
            available_columns = df.columns.tolist()
            raise ValueError(
                "Не найдены обязательные колонки: "
                f"{missing_columns}. Доступные колонки после нормализации: {available_columns}"
            )

    def load_source_data(self) -> pd.DataFrame:
        values = self.source_connect.sheet_title.get_all_values()
        if len(values) < 4:
            raise ValueError("В листе меньше 4 строк, строка заголовков не найдена.")

        headers = self.make_unique_column_names(
            [self.normalize_column_name(header) for header in values[3]]
        )
        df = pd.DataFrame(values[4:], columns=headers)

        for column in ORDERS_WHITE_DIGIT_COLS:
            df[column] = df[column].apply(clean_currency_value)

        return df

    def prepare_orders_dataframe(self, df: pd.DataFrame) -> pd.DataFrame:
        self.validate_required_columns(df, ORDERS_WHITE_REQUIRED_COLUMNS)

        df_orders = df.loc[:, ORDERS_WHITE_REQUIRED_COLUMNS].copy()
        return df_orders[~df_orders["Статус"].isin(ORDERS_WHITE_CANCELED_STATUSES)]

    @staticmethod
    def get_order_id_columns() -> list[str]:
        payment_columns = (
            ORDERS_WHITE_AVANS_COLS
            + ORDERS_WHITE_BALANCE_ONE_COLS
            + ORDERS_WHITE_BALANCE_TWO_COLS
        )
        return [
            column
            for column in ORDERS_WHITE_REQUIRED_COLUMNS
            if column not in payment_columns
        ]

    def build_payment_dataframe(
        self,
        df: pd.DataFrame,
        base_columns: list[str],
        payment_config: dict[str, object],
    ) -> pd.DataFrame:
        payment_columns = payment_config["columns"]
        self.validate_required_columns(df, base_columns + list(payment_columns.values()))

        df_payment = df.loc[:, base_columns].copy()
        df_payment["_Порядок исходной строки"] = range(len(df))
        df_payment["Этап платежа"] = payment_config["Этап платежа"]
        df_payment["Номер этапа платежа"] = payment_config["Номер этапа платежа"]

        for target_column, source_column in payment_columns.items():
            df_payment[target_column] = df[source_column].to_numpy()

        return df_payment

    def build_balance_dataframe(self, df_orders: pd.DataFrame) -> pd.DataFrame:
        base_columns = self.get_order_id_columns()
        payment_frames = [
            self.build_payment_dataframe(
                df=df_orders,
                base_columns=base_columns,
                payment_config=payment_config,
            )
            for payment_config in ORDERS_WHITE_PAYMENT_CONFIGS
        ]

        df_balance = pd.concat(payment_frames, ignore_index=True)
        df_balance = df_balance[
            df_balance["Статус_по_этапу"].notna()
        ].copy()

        return df_balance.sort_values(
            by=["Номер заказа 1С", "_Порядок исходной строки", "Номер этапа платежа"],
            kind="stable",
        ).drop(columns="_Порядок исходной строки").reset_index(drop=True)

    @staticmethod
    def add_payment_status_amounts(df_balance: pd.DataFrame) -> pd.DataFrame:
        df_result = df_balance.copy()
        paid_mask = df_result["Статус_по_этапу"].eq("оплачено")
        payment_amount = df_result["Сумма_оплаты"].fillna(0)

        df_result["Оплачено"] = payment_amount.where(paid_mask, 0)
        df_result["Не_оплачено"] = payment_amount.where(~paid_mask, 0)
        return df_result

    @staticmethod
    def prepare_dataframe_for_upload(df_balance: pd.DataFrame) -> pd.DataFrame:
        df_upload = df_balance.copy()
        df_upload[ORDERS_WHITE_UPDATED_AT_COLUMN] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        return df_upload

    def upload_to_google_sheet(self, df_upload: pd.DataFrame) -> None:
        if df_upload.empty:
            logger.warning("df_orders_white_balance пустой. Выгрузка в Google Sheets пропущена.")
            return

        self.target_connect.set_df_to_google(df_upload)
        logger.info("df_orders_white_balance выгружен на лист %s.", self._target_sheet_name)

    def run(self) -> pd.DataFrame:
        df_source = self.load_source_data()
        df_orders = self.prepare_orders_dataframe(df_source)
        df_balance = self.build_balance_dataframe(df_orders)
        df_balance = self.add_payment_status_amounts(df_balance)
        df_upload = self.prepare_dataframe_for_upload(df_balance)
        # self.upload_to_google_sheet(df_upload)
        return df_balance


In [4]:
table = OrdersWhiteBalanceAnalyticsService()
balance_df = table.run()

✅ Успешное подключение к Расчет поставки Китай_по обороту -> Заказы белые ТЕСТ


In [2]:
import logging
import re

import pandas as pd

from src_oop.core.my_gspread import GoogleTabs
from src_oop.core.utils_general import clean_currency_value
from src_oop.jobs.calculation_of_purchases_china.config import (
    LOGISTICS_VED_REQUIRED_COLUMNS,
    VED_DIGIT_COLS,
    VED_PAYMENT_CONFIGS,
    delivery_calculation_china,
    logistics_ved,
)

logger = logging.getLogger(__name__)


class LogisticsVED:
    """??????? ???????? ????????? VED ? ???????, ??????????? ? balance_df."""

    BASE_COLUMNS_MAP = {
        "??????": "????????",
        "? ????????": "? ???????? ? ?????????? ? 1?",
        "??????": "??????",
        "????????": "wild",
        "???????????? ??????": "??????",
        "?? ????": "????????",
        "?????????? ????": "???-?? ? ??????",
        "???????????? ????????": "?????????",
    }

    DEFAULT_COLUMNS = {
        "????? ?????? 1?": pd.NA,
        "??? ??????": pd.NA,
        "????_??????_??_????????_?????": pd.NA,
        "????_????": pd.NA,
        "%_??????": pd.NA,
    }

    def __init__(self) -> None:
        self._table_name = logistics_ved.get("title")
        self._source_sheet_name = logistics_ved.get("test_sheet")
        self._source_conn: GoogleTabs | None = None

    @property
    def source_connect(self) -> GoogleTabs:
        if self._source_conn is None:
            self._source_conn = GoogleTabs(
                table_title=self._table_name,
                sheet_title=self._source_sheet_name,
            )
        return self._source_conn

    @staticmethod
    def normalize_column_name(column_name: object) -> str:
        normalized = str(column_name).replace("\t", " ").replace("\n", " ").replace("\r", " ")
        return re.sub(r"\s+", " ", normalized).strip()

    @staticmethod
    def make_unique_column_names(column_names: list[str]) -> list[str]:
        seen_columns: dict[str, int] = {}
        unique_columns: list[str] = []

        for column_name in column_names:
            seen_columns[column_name] = seen_columns.get(column_name, 0) + 1
            if seen_columns[column_name] == 1:
                unique_columns.append(column_name)
                continue

            unique_columns.append(f"{column_name} {seen_columns[column_name]}")

        return unique_columns

    @staticmethod
    def validate_required_columns(df: pd.DataFrame, required_columns: list[str]) -> None:
        missing_columns = [column for column in required_columns if column not in df.columns]
        if missing_columns:
            available_columns = df.columns.tolist()
            raise ValueError(
                "?? ??????? ???????????? ???????: "
                f"{missing_columns}. ????????? ??????? ????? ????????????: {available_columns}"
            )

    def load_source_data(self) -> pd.DataFrame:
        values = self.source_connect.sheet_title.get_all_values()
        if len(values) < 3:
            raise ValueError("? ????? ?????? 3 ?????, ?????? ?????????? ?? ???????.")

        headers = self.make_unique_column_names(
            [self.normalize_column_name(header) for header in values[2]]
        )
        df = pd.DataFrame(values[3:], columns=headers)

        for column in VED_DIGIT_COLS:
            if column in df.columns:
                df[column] = df[column].apply(clean_currency_value)

        return df

    def prepare_source_dataframe(self, df: pd.DataFrame) -> pd.DataFrame:
        self.validate_required_columns(df, LOGISTICS_VED_REQUIRED_COLUMNS)
        self.validate_required_columns(df, list(self.BASE_COLUMNS_MAP.keys()))

        df_orders = df.loc[:, LOGISTICS_VED_REQUIRED_COLUMNS].copy()
        df_orders = df_orders.rename(columns=self.BASE_COLUMNS_MAP)

        for column_name, default_value in self.DEFAULT_COLUMNS.items():
            if column_name not in df_orders.columns:
                df_orders[column_name] = default_value

        return df_orders

    def build_payment_dataframe(
        self,
        df_source: pd.DataFrame,
        df_base: pd.DataFrame,
        payment_config: dict[str, object],
    ) -> pd.DataFrame:
        payment_columns = payment_config["columns"]
        self.validate_required_columns(df_source, list(payment_columns.values()))

        df_payment = df_base.copy()
        df_payment["_??????? ???????? ??????"] = range(len(df_payment))
        df_payment["???? ???????"] = payment_config["???? ???????"]
        df_payment["????? ????? ???????"] = payment_config["????? ????? ???????"]

        for target_column, source_column in payment_columns.items():
            df_payment[target_column] = df_source[source_column].to_numpy()

        if "?????_??????" not in df_payment.columns:
            df_payment["?????_??????"] = pd.NA
        if "????_????" not in df_payment.columns:
            df_payment["????_????"] = pd.NA
        if "????_??????_?????????" not in df_payment.columns:
            df_payment["????_??????_?????????"] = pd.NA
        if "??????_??_?????" not in df_payment.columns:
            df_payment["??????_??_?????"] = pd.NA

        df_payment["????_??????_??_????????_?????"] = pd.NA
        df_payment["????_????"] = pd.NA
        df_payment["%_??????"] = pd.NA
        return df_payment

    def build_balance_dataframe(self, df_source: pd.DataFrame) -> pd.DataFrame:
        df_base = self.prepare_source_dataframe(df_source)
        payment_frames = [
            self.build_payment_dataframe(
                df_source=df_source,
                df_base=df_base,
                payment_config=payment_config,
            )
            for payment_config in VED_PAYMENT_CONFIGS
        ]

        df_balance = pd.concat(payment_frames, ignore_index=True)
        status_series = df_balance["??????_??_?????"].fillna("").astype(str).str.strip()
        df_balance = df_balance[status_series.ne("")].copy()

        return df_balance.sort_values(
            by=["? ???????? ? ?????????? ? 1?", "_??????? ???????? ??????", "????? ????? ???????"],
            kind="stable",
        ).drop(columns="_??????? ???????? ??????").reset_index(drop=True)

    @staticmethod
    def add_payment_status_amounts(df_balance: pd.DataFrame) -> pd.DataFrame:
        df_result = df_balance.copy()
        paid_mask = (
            df_result["??????_??_?????"]
            .fillna("")
            .astype(str)
            .str.strip()
            .str.lower()
            .eq("????????")
        )
        payment_amount = pd.to_numeric(df_result["?????_??????"], errors="coerce").fillna(0)

        df_result["????????"] = payment_amount.where(paid_mask, 0)
        df_result["??_????????"] = payment_amount.where(~paid_mask, 0)
        return df_result

    def align_to_balance_columns(self, ved_balance_df: pd.DataFrame, balance_columns: list[str]) -> tuple[pd.DataFrame, list[str], list[str]]:
        missing_columns = [column for column in balance_columns if column not in ved_balance_df.columns]
        extra_columns = [column for column in ved_balance_df.columns if column not in balance_columns]

        df_aligned = ved_balance_df.copy()
        for column in missing_columns:
            df_aligned[column] = pd.NA

        return df_aligned.loc[:, balance_columns], missing_columns, extra_columns

    @staticmethod
    def build_duplicate_risk_report(ved_balance_df: pd.DataFrame) -> pd.DataFrame:
        risk_columns = [
            "? ???????? ? ?????????? ? 1?",
            "wild",
            "??????",
            "??????_??_?????",
            "????_????",
            "????_??????_?????????",
            "?????_??????",
        ]
        available_columns = [column for column in risk_columns if column in ved_balance_df.columns]
        if not available_columns:
            return pd.DataFrame()

        df_risk = ved_balance_df[ved_balance_df["????? ????? ???????"].isin([5, 6])].copy()
        if df_risk.empty:
            return pd.DataFrame(columns=available_columns + ["?????????? ??????"])

        return (
            df_risk.groupby(available_columns, dropna=False)
            .size()
            .reset_index(name="?????????? ??????")
            .query("`?????????? ??????` > 1")
            .sort_values("?????????? ??????", ascending=False)
            .reset_index(drop=True)
        )

    def run(self) -> pd.DataFrame:
        df_source = self.load_source_data()
        df_balance = self.build_balance_dataframe(df_source)
        return self.add_payment_status_amounts(df_balance)


In [3]:
ved_df = LogisticsVED()
ved_balance_df = ved_df.run()
ved_balance_df.head()

✅ Успешное подключение к Логистика ВЭД 2026 -> Виктория


ValueError: ?? ??????? ???????????? ???????: ['??????', '? ????????', '????????', '???????????? ??????', '?? ????', '?????????? ????', '???????????? ????????']. ????????? ??????? ????? ????????????: ['Логист', 'Номер заказа в 1С', '№ ПРОФОРМЫ', 'Статус', '№ ТС', 'АРТИКУЛЫ', 'НАИМЕНОВАНИЕ ТОВАРА', 'ПОСТАВЩИК', 'Юр лицо', 'УСЛОВИЯ ОПЛАТЫ', 'КОЛИЧЕСТВО ШТУК', 'КОЛИЧЕСТВО КОРОБОВ', 'ОБЪЕМ , М3', 'ОПЛАТА ТОВАРА', 'ДОКУМЕНТЫ ОТ ОТПРАВИТЕЛЯ ПОЛУЧЕНЫ', 'EX-1 получена', 'ГОТОВНОСТЬ РАЗРЕШИТЕЛЬНЫХ ДОКУМЕНТОВ', 'ГОТОВНОСТЬ СТРАХОВОГО ПОЛИСА', 'СТАТУС ОТПРАВКИ ДОКУМЕНТОВ ТАМОЖЕННОМУ БРОКЕРУ', 'ДОБАВЛЕНО В 1С', 'ДОБАВЛЕНО В ФАЙЛ РАСЧЕТЫ КИТАЙ', 'ДАТА ПЕРЕВОДА ТП НА ЕЛС', 'ПЛАНОВАЯ ДАТА ОТГРУЗКИ ОТ ПОСТАВЩИКА', 'ФАКТИЧЕСКАЯ ДАТА ОТГРУЗКИ ОТ ПОСТАВЩИКА', 'ДАТА ПРИБЫТИЯ ТС НА ПЕРЕГРУЗКУ', 'Количество дней до границы', 'ДАТА ВЫХОДА ТС С ПЕРЕГРУЗКИ', 'Количество дней на перегрузке', 'ПЛАНОВАЯ ДАТА ПРИБЫТИЯ НА ТАМОЖНЮ', 'ФАКТИЧЕСКАЯ ДАТА ПРИБЫТИЯ В ПОРТ/ НА СТАНЦИЮ НАЗНАЧЕНИЯ / На Таможню', 'Количество д

In [ ]:
balance_columns = balance_df.columns.tolist()
ved_columns = ved_balance_df.columns.tolist()

missing_in_ved = [column for column in balance_columns if column not in ved_columns]
extra_in_ved = [column for column in ved_columns if column not in balance_columns]

ved_balance_aligned_df, auto_added_columns, ved_extra_columns = ved_df.align_to_balance_columns(
    ved_balance_df=ved_balance_df,
    balance_columns=balance_columns,
)
combined_balance_df = pd.concat([balance_df, ved_balance_aligned_df], ignore_index=True)

stage_counts = (
    ved_balance_df.groupby(["????? ????? ???????", "???? ???????"], dropna=False)
    .size()
    .reset_index(name="?????????? ?????")
    .sort_values(["????? ????? ???????", "???? ???????"])
)

stage_amounts = (
    ved_balance_df.groupby(["????? ????? ???????", "???? ???????"], dropna=False)["?????_??????"]
    .sum(min_count=1)
    .reset_index(name="?????_??????_?????")
    .sort_values(["????? ????? ???????", "???? ???????"])
)

duplicate_risk_df = ved_df.build_duplicate_risk_report(ved_balance_df)

print("balance_df shape:", balance_df.shape)
print("ved_balance_df shape:", ved_balance_df.shape)
print("ved_balance_aligned_df shape:", ved_balance_aligned_df.shape)
print("combined_balance_df shape:", combined_balance_df.shape)
print("missing_in_ved:", missing_in_ved)
print("extra_in_ved:", extra_in_ved)
print("auto_added_columns:", auto_added_columns)
print("ved_extra_columns:", ved_extra_columns)
print("duplicate_risk_rows_for_stages_5_6:", duplicate_risk_df.shape[0])


In [ ]:
display(stage_counts)
display(stage_amounts)
display(duplicate_risk_df.head(20))
display(ved_balance_aligned_df.head())
display(combined_balance_df.head())


In [ ]:
test_sheet_conn = GoogleTabs(
    table_title=delivery_calculation_china.get("title"),
    sheet_title=delivery_calculation_china.get("test_sheet"),
)

# ?????????? ???????? ???????? ?????? ? delivery_calculation_china / test_sheet.
# ?????????????? ?????? ???? ????? ?????????? ???????? combined_balance_df.
# test_sheet_conn.set_df_to_google(combined_balance_df)
